In [ ]:
import pandas as pd

from gsm_benchmarker.results_analyser.utils import pandas_to_latex, format_p_value, format_float
from results_notebook_setup import load_results, significant_models

In [2]:
N_BOOT = 2000
results_loader = load_results(n_boot=N_BOOT)

In [ ]:
short_code_result = results_loader.short_code
short_code_result_clean = results_loader.short_code_filtered

long_code_result = results_loader.long_code
long_code_result_clean = results_loader.long_code_filtered


In [4]:
vek = {'model_order': significant_models}

In [5]:
def match_index(s, df):
    if not isinstance(df.index, pd.MultiIndex) or df.index.nlevels < 2:
        return s

    if df.index.nlevels > 2:
        raise RuntimeError('Matching dataframe index for a multi-index of more than 2 levels not implemented')

    index_names = df.index.names
    s_matched = pd.DataFrame({
        k: s for k in df.index.get_level_values(index_names[1]).unique()
    }).stack()
    s_matched.index.names = index_names
    return s_matched


def compare(orig_res, clean_res, glmm_id, alpha=0.05, model_order: list[str] | None = None, metric: str | None = None):

    orig_df, clean_df = [getattr(res, f'glmm{glmm_id}_results') for res in (orig_res, clean_res)]

    p_label = "P value"
    acc_diff_label = r"$\Delta_{var}$"
    excl_rate_label = "Exclusion rate"
    est_delta_label = r'Estimate $\Delta$'

    report_dfs = []
    for res in (orig_res, clean_res):
        df = getattr(res, f'glmm{glmm_id}_results')
        d = {acc_diff_label: df.acc_diff} if glmm_id == '1' else {}
        report_df = pd.DataFrame({
            **d,
            p_label: df.boot_p_value,
        }, index=df.index)
        report_dfs.append(report_df)

    cat_labels = ['All errors', 'Reasoning-only']
    comparison_df = pd.concat(report_dfs, axis=1, keys=cat_labels)

    n_resp, n_clean_resp = [res.mres.variants['main'].full_data.groupby('model').size() for res in (orig_res, clean_res)]
    er = 1 - n_clean_resp / n_resp

    comparison_df[('', excl_rate_label)] = 100 * match_index(er, comparison_df)
    comparison_df[('', est_delta_label)] = clean_df['boot_median_log'] - orig_df['boot_median_log']

    if model_order is not None:
        comparison_df.sort_index(
            level='model',
            key=lambda idx: idx.map({model: i for i, model in enumerate(model_order)}),
            inplace=True
        )
        comparison_df = comparison_df[comparison_df.index.isin(model_order, level='model')]

    comparison_df.index.names = ('Model', 'Metric')
    if metric:
        comparison_df = comparison_df.xs(metric, level='Metric')
    comparison_df = comparison_df.reset_index()

    agreement = (comparison_df[(cat_labels[0], p_label)] < alpha) == (comparison_df[(cat_labels[1], p_label)] < alpha)
    comparison_df[('Model', '')] = [m + ("" if a else r" \textbf{*}") for m, a in zip(comparison_df[('Model', '')], agreement)]

    pf = format_p_value(3, use_delta=True)
    for cat in cat_labels:
        comparison_df[(cat, p_label)] = comparison_df[(cat, p_label)].apply(pf)
        if glmm_id == '1':
            comparison_df[(cat, acc_diff_label)] = comparison_df[(cat, acc_diff_label)].apply(format_float(1, use_abs=True))
    comparison_df[('', excl_rate_label)] = comparison_df[('', excl_rate_label)].apply(format_float(1))
    comparison_df[('', est_delta_label)] = comparison_df[('', est_delta_label)].apply(format_float(3, use_abs=True))
    if glmm_id != '1':
        comparison_df[('Metric', '')] = comparison_df[('Metric', '')].apply(lambda s: ("$Variant$" if s == 'is_variant' else r"$\gamma_c$"))



    print(f"\nAgreement rate: {agreement.sum()}/{len(agreement)}")
    print()

    print(pandas_to_latex(
        comparison_df,
        index=False,
        caption=f"GLMM {glmm_id}",
        position='H',
        multicolumn_format='c',
        multicolumn=True,
        clean_header=False,
        column_format=f'l{"" if metric else "c"}|cc|cc|cc'
    ))



In [6]:
compare(short_code_result, short_code_result_clean, '1', **vek, metric='is_variant')


Agreement rate: 6/8

\begin{table}[H]
\caption{GLMM 1}
\begin{tabular}{l|cc|cc|cc}
\toprule
Model & \multicolumn{2}{c}{All errors} & \multicolumn{2}{c}{Reasoning-only} & \multicolumn{2}{c}{} \\
 & $\Delta_{var}$ & P value & $\Delta_{var}$ & P value & Exclusion rate & Estimate $\Delta$ \\
\midrule
phi-2 & 3.6 & 0.179 & 4.1 & 0.133 & 0.8 & 0.065 \\
Phi-3.5-mini-instruct \textbf{*} & -4.4 & \textbf{0.050} & -4.2 & 0.067 & 0.5 & 0.035 \\
gemma-2b & -0.6 & 0.835 & -0.3 & 0.924 & 9.5 & 0.049 \\
gemma-2-9b & -2.2 & 0.416 & -2.0 & 0.460 & 1.7 & 0.015 \\
Mathstral-7B-v0.1 & 3.9 & 0.241 & 3.6 & 0.258 & 1.4 & -0.016 \\
Meta-Llama-3-8B \textbf{*} & 7.1 & 0.050 & 8.5 & \textbf{0.017} & 5.4 & 0.150 \\
gemma-7b-it & -6.4 & 0.066 & -5.8 & 0.107 & 7.3 & 0.007 \\
Mistral-7B-Instruct-v0.1 & 2.4 & 0.512 & 2.9 & 0.460 & 6.5 & 0.072 \\
\bottomrule
\end{tabular}
\end{table}



In [7]:
compare(long_code_result, long_code_result_clean, '1', **vek, metric='is_variant')


Agreement rate: 8/8

\begin{table}[H]
\caption{GLMM 1}
\begin{tabular}{l|cc|cc|cc}
\toprule
Model & \multicolumn{2}{c}{All errors} & \multicolumn{2}{c}{Reasoning-only} & \multicolumn{2}{c}{} \\
 & $\Delta_{var}$ & P value & $\Delta_{var}$ & P value & Exclusion rate & Estimate $\Delta$ \\
\midrule
phi-2 & -2.2 & 0.457 & -2.6 & 0.383 & 0.5 & -0.060 \\
Phi-3.5-mini-instruct & -2.9 & 0.268 & -2.7 & 0.306 & 0.5 & 0.033 \\
gemma-2b & -2.6 & 0.359 & -2.4 & 0.417 & 8.4 & 0.044 \\
gemma-2-9b & $|\cdot|$ < 0.1 & 0.973 & -0.4 & 0.858 & 1.7 & -0.075 \\
Mathstral-7B-v0.1 & 1.6 & 0.547 & 2.0 & 0.394 & 2.2 & 0.090 \\
Meta-Llama-3-8B & 0.9 & 0.774 & 1.6 & 0.603 & 3.3 & 0.073 \\
gemma-7b-it & -2.4 & 0.540 & -2.9 & 0.485 & 7.4 & -0.051 \\
Mistral-7B-Instruct-v0.1 & -2.9 & 0.392 & -3.5 & 0.313 & 7.8 & -0.069 \\
\bottomrule
\end{tabular}
\end{table}



In [8]:
compare(short_code_result, short_code_result_clean, glmm_id='2', **vek)


Agreement rate: 14/16

\begin{table}[H]
\caption{GLMM 2}
\begin{tabular}{lc|cc|cc|cc}
\toprule
Model & Metric & All errors & Reasoning-only & \multicolumn{2}{c}{} \\
 &  & P value & P value & Exclusion rate & Estimate $\Delta$ \\
\midrule
phi-2 & $\gamma_c$ & 0.480 & 0.446 & 0.8 & -0.011 \\
phi-2 & $Variant$ & 0.141 & 0.104 & 0.8 & 0.082 \\
Phi-3.5-mini-instruct & $\gamma_c$ & 0.828 & 0.832 & 0.5 & $|\cdot|$ < 0.001 \\
Phi-3.5-mini-instruct \textbf{*} & $Variant$ & \textbf{0.045} & 0.062 & 0.5 & 0.026 \\
gemma-2b & $\gamma_c$ & 0.778 & 0.779 & 9.5 & -0.002 \\
gemma-2b & $Variant$ & 0.909 & 0.984 & 9.5 & 0.061 \\
gemma-2-9b & $\gamma_c$ & 0.270 & 0.345 & 1.7 & -0.017 \\
gemma-2-9b & $Variant$ & 0.216 & 0.246 & 1.7 & 0.051 \\
Mathstral-7B-v0.1 & $\gamma_c$ & 0.266 & 0.257 & 1.4 & -0.012 \\
Mathstral-7B-v0.1 & $Variant$ & 0.184 & 0.197 & 1.4 & -0.012 \\
Meta-Llama-3-8B & $\gamma_c$ & 0.305 & 0.524 & 5.4 & -0.046 \\
Meta-Llama-3-8B \textbf{*} & $Variant$ & 0.093 & \textbf{0.028} & 5.4 & 0

In [9]:
compare(long_code_result, long_code_result_clean, glmm_id='2', **vek)



Agreement rate: 16/16

\begin{table}[H]
\caption{GLMM 2}
\begin{tabular}{lc|cc|cc|cc}
\toprule
Model & Metric & All errors & Reasoning-only & \multicolumn{2}{c}{} \\
 &  & P value & P value & Exclusion rate & Estimate $\Delta$ \\
\midrule
phi-2 & $\gamma_c$ & 0.981 & 0.972 & 0.5 & $|\cdot|$ < 0.001 \\
phi-2 & $Variant$ & 0.480 & 0.378 & 0.5 & -0.067 \\
Phi-3.5-mini-instruct & $\gamma_c$ & 0.232 & 0.243 & 0.5 & 0.004 \\
Phi-3.5-mini-instruct & $Variant$ & 0.483 & 0.536 & 0.5 & 0.029 \\
gemma-2b & $\gamma_c$ & 0.731 & 0.640 & 8.4 & -0.015 \\
gemma-2b & $Variant$ & 0.413 & 0.499 & 8.4 & 0.074 \\
gemma-2-9b & $\gamma_c$ & 0.538 & 0.570 & 1.7 & 0.003 \\
gemma-2-9b & $Variant$ & 0.984 & 0.962 & 1.7 & -0.040 \\
Mathstral-7B-v0.1 & $\gamma_c$ & 0.645 & 0.902 & 2.2 & 0.036 \\
Mathstral-7B-v0.1 & $Variant$ & 0.637 & 0.602 & 2.2 & 0.029 \\
Meta-Llama-3-8B & $\gamma_c$ & 0.683 & 0.683 & 3.3 & 0.003 \\
Meta-Llama-3-8B & $Variant$ & 0.699 & 0.544 & 3.3 & 0.080 \\
gemma-7b-it & $\gamma_c$ & 0.308 & 